<a href="https://colab.research.google.com/github/Fatima-TulZahra/AI-Dataset-Advisor/blob/ml%2Fmeta-model/02_download_and_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/ai_dataset_advisor'
for d in ['data/clf', 'data/reg', 'results', 'artifacts', 'openml_cache']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q openml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 47.5 MB/s eta 0:00:00


In [ ]:
import requests, time

tests = {
    "api.openml.org (json)": "https://api.openml.org/api/v1/json/data/3",
    "www.openml.org (json)": "https://www.openml.org/api/v1/json/data/3",
    "www.openml.org (site)": "https://www.openml.org",
    "github (control)":      "https://raw.githubusercontent.com",
    "google (control)":      "https://www.google.com",
}
for name, url in tests.items():
    t0 = time.time()
    try:
        r = requests.get(url, timeout=15)
        print(f"{name:24} -> {r.status_code}   {time.time()-t0:.1f}s")
    except Exception as e:
        print(f"{name:24} -> FAIL ({type(e).__name__})   {time.time()-t0:.1f}s")

api.openml.org (json)    -> 200   4.2s
www.openml.org (json)    -> 200   1.6s
www.openml.org (site)    -> 200   0.3s
github (control)         -> 200   0.2s
google (control)         -> 200   0.1s


In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd

lst = pd.read_csv(f"{BASE}/datasets_list.csv")

def download(did, task):
    b = fetch_openml(data_id=int(did), as_frame=True, data_home=f"{BASE}/openml_cache")
    X, y = b.data.copy(), b.target
    ok = y.notna()
    X, y = X[ok], y[ok]                       # target jis row mein khali ho wo hata do
    X = X.dropna(axis=1, how="all")           # jis column mein sab khali ho wo hata do
    if len(X) > 5000:                         # tez experiments ke liye sample
        idx = X.sample(5000, random_state=42).index
        X, y = X.loc[idx], y.loc[idx]
    df = X.copy(); df["__target__"] = y.values
    folder = "clf" if task == "classification" else "reg"
    df.to_csv(f"{BASE}/data/{folder}/{did}.csv", index=False)

failed = []
for r in lst.itertuples():
    try:
        download(r.did, r.task)
    except Exception as e:
        failed.append((r.did, str(e)[:100]))
pd.DataFrame(failed, columns=["did", "error"]).to_csv(f"{BASE}/results/failed_downloads.csv", index=False)
print("failed:", len(failed))

/usr/local/lib/python3.13/dist-packages/sklearn/datasets/_openml.py:110: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/features/44540. Retrying...
  warn(


failed: 13


In [ ]:
import sys; sys.path.append(f"{BASE}/code")
from meta_features import extract_meta_features
import pandas as pd, os

fname = sorted(os.listdir(f"{BASE}/data/clf"))[0]
df = pd.read_csv(f"{BASE}/data/clf/{fname}")
y = df.pop("__target__")
print(fname, extract_meta_features(df, y, "classification"))

1000.csv {'n_rows': 3772, 'n_cols': 28, 'log_n_rows': 3.5765716840652906, 'rows_per_col': 134.71428571428572, 'numeric_ratio': 0.21428571428571427, 'categorical_ratio': 0.7857142857142857, 'missing_pct': 2.170125738524466, 'mean_abs_correlation': 0.2751391822207439, 'mean_cardinality': 2.090909090909091, 'n_classes': 2, 'class_imbalance': 0.8457051961823967}


In [ ]:
import os, pandas as pd
lst = pd.read_csv(f"{BASE}/datasets_list.csv")
lst["folder"] = lst.task.map({"classification": "clf", "regression": "reg"})
lst["exists"] = [os.path.exists(f"{BASE}/data/{f}/{d}.csv") for f, d in zip(lst.folder, lst.did)]
print(lst.groupby("task").exists.agg(["sum", "count"]))
print(len(lst[~lst.exists]), "files abhi baqi hain")

                sum  count
task                      
classification  246    249
regression      110    120
13 files abhi baqi hain
